# Combined classifier features with frozen DBSCAN labels

Test the four previously proposed combinations using logistic regression: **diseases + criteria**, **diseases + pathways**, **criteria + pathways**, and **all three**. Use **C=100 and C=1000**, plus an embeddings-only control at each C: **10 configurations**.

Freeze preprocessing to the existing representations: diseases appearing in at least 5 training trials (**71 indicators**, the smallest tied disease representation), criteria TF-IDF with minimum count 3 (**2,000 terms**, the best previous text representation), and broad level-0 pathways with minimum count 30 (**26 indicators + missing flag**, the smallest tied pathway representation). The embeddings are included exactly once. All added feature-block weights remain 1.

Targets, pseudo-label weights, embedding scaler, vocabularies, IDF, classifier fitting-weight total 157 per reason, cutoff 0.5, and highest-probability fallback stay fixed. No clustering, preprocessing refit, or nonlinear models in this round. Save the combined matrices and preprocessing objects for a future nonlinear comparison. No Kaggle submission, commit, or tag.

In [1]:
from pathlib import Path
import ast
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score
from threadpoolctl import threadpool_limits
from IPython.display import display

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
EMBEDDINGS=[f"emb_{i}" for i in range(150)]
C_GRID=[100.0,1000.0]
BLOCK_WEIGHTS=[1.0]
COMBINATIONS={"embeddings_only":[],
    "diseases_criteria":["diseases_min5","criteria_min3"],
    "diseases_pathways":["diseases_min5","pathways_min30"],
    "criteria_pathways":["criteria_min3","pathways_min30"],
    "diseases_criteria_pathways":["diseases_min5","criteria_min3","pathways_min30"]}
REPRESENTATIONS=[name for name in COMBINATIONS if name!="embeddings_only"]
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
SOURCE=DATA/"experiments/classifier_features"
OUT=DATA/"experiments/classifier_combinations"
for folder in ["datasets","models","validation_predictions","submissions"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv")
valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
truth=valid[REASONS].to_numpy(dtype=int)
original_y=train[REASONS].to_numpy(dtype=int)
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert len(train)==1171 and len(valid)==78 and known.sum()==157
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
fixed=pd.read_csv(SOURCE/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed.ID.tolist()==train.ID.tolist()
Y=fixed[REASONS].to_numpy(dtype=int);W=fixed[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original_y[known]);np.testing.assert_array_equal(W[known],np.ones((157,5)))
assert np.array_equal(W>0,Y!=-100)
(OUT/"datasets/fixed_targets_weights.csv").write_bytes((SOURCE/"datasets/fixed_targets_weights.csv").read_bytes())
source_settings=json.loads((SOURCE/"settings.json").read_text())
reference_sample=float(source_settings["reference_sample_f1"])
reference_macro=float(source_settings["reference_macro_f1"])
source_names=["embeddings_only","diseases_min5","criteria_min3","pathways_min30"]
ORIGINAL_MODELS={name:joblib.load(SOURCE/"models"/f"{name}.joblib") for name in source_names}
SOURCE_MODELS={"embeddings_only":ORIGINAL_MODELS["embeddings_only"]}
for name,parts in COMBINATIONS.items():
    if not parts:continue
    blocks=[ORIGINAL_MODELS[part]["block"] for part in parts]
    SOURCE_MODELS[name]={**ORIGINAL_MODELS["embeddings_only"],
                        "block":{"kind":"combination","components":blocks,
                                 "feature_names":[v for block in blocks for v in block["feature_names"]]}}
    assert len(set(SOURCE_MODELS[name]["block"]["feature_names"]))==len(SOURCE_MODELS[name]["block"]["feature_names"])
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",SOURCE/"settings.json",SOURCE/"comparison.csv",SOURCE/"datasets/fixed_targets_weights.csv"]
for name in source_names:protected_paths.extend([SOURCE/"models"/f"{name}.joblib",SOURCE/"datasets"/f"{name}.npz"])
additional=DATA/"experiments/additional_methods"
for rel in ["datasets/dbscan_m5_q25.csv","labeling_models/dbscan_m5_q25.joblib","classifier_models/dbscan_m5_q25__logistic_C1000.joblib"]:
    protected_paths.append(additional/rel)
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("Frozen targets, weights, embeddings, and vocabularies; 10 classifier settings declared.")


Frozen targets, weights, embeddings, and vocabularies; 10 classifier settings declared.


In [2]:
def disease_lists(frame):
    def parse(value):
        if pd.isna(value) or not str(value).strip():return []
        values=ast.literal_eval(value)
        assert isinstance(values,list)
        return sorted({str(v).strip().lower() for v in values if str(v).strip()})
    return frame.diseases_cleaned.map(parse).tolist()


def pathway_lists(frame):
    return frame.drug_pathways.fillna("").map(lambda s:sorted({v.strip() for v in str(s).split(";") if v.strip()})).tolist()


def missing_pathways(frame):
    return frame.drug_pathways.fillna("").astype(str).str.strip().eq("").to_numpy(dtype=float)[:,None]


def transform_block(block,frame):
    kind=block["kind"]
    if kind=="combination":return sparse.hstack([transform_block(component,frame) for component in block["components"]],format="csr")
    if kind=="none":return sparse.csr_matrix((len(frame),0))
    if kind=="missing_only":return sparse.csr_matrix(missing_pathways(frame))
    if kind=="criteria":return block["vectorizer"].transform(frame.criteria.fillna("")).tocsr()
    values=disease_lists(frame) if kind=="diseases" else pathway_lists(frame)
    allowed=set(block["encoder"].classes_)
    result=block["encoder"].transform([[v for v in row if v in allowed] for row in values]).astype(float).tocsr()
    if kind=="pathways":result=sparse.hstack([result,sparse.csr_matrix(missing_pathways(frame))],format="csr")
    assert result.shape[1]==len(block["feature_names"])
    assert np.isin(result.data,[0,1]).all()
    return result


def transform_model(model,frame):
    embeddings=model["embedding_scaler"].transform(frame[model["embedding_features"]].to_numpy())
    if model["block"]["kind"]=="none":return embeddings
    return sparse.hstack([sparse.csr_matrix(embeddings),transform_block(model["block"],frame)*model.get("feature_block_weight",1.0)],format="csr")


def predict_model(model,frame):
    matrix=transform_model(model,frame)
    return np.column_stack([est.predict_proba(matrix)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    predicted=(probabilities>=0.5).astype(int)
    empty=predicted.sum(axis=1)==0
    rows=np.flatnonzero(empty);predicted[rows,probabilities[rows].argmax(axis=1)]=1
    return predicted,empty

reference_probability=predict_model(SOURCE_MODELS["embeddings_only"],valid)
reference_prediction,_=label_predictions(reference_probability)
np.testing.assert_allclose(f1_score(truth,reference_prediction,average="samples",zero_division=0),reference_sample,atol=1e-14)
ROWS=[];MATRIX_RECORDS={};BASELINE_SCORES={}


def prepare_matrix(representation,block_weight):
    assert block_weight==1.0
    source_model=SOURCE_MODELS[representation]
    base=sparse.load_npz(SOURCE/"datasets/embeddings_only.npz")
    parts=COMBINATIONS[representation]
    blocks=[]
    for part in parts:
        matrix=sparse.load_npz(SOURCE/"datasets"/f"{part}.npz")
        difference=matrix[:,:150]-base
        assert not difference.nnz or abs(difference.data).max()<1e-12
        blocks.append(matrix[:,150:])
    matrix=sparse.hstack([base]+blocks,format="csr") if blocks else base.copy()
    expected=sparse.csr_matrix(transform_model(source_model,train))
    difference=matrix-expected
    assert not difference.nnz or abs(difference.data).max()<1e-12
    names=EMBEDDINGS+source_model["block"]["feature_names"]
    assert matrix.shape==(len(train),len(names)) and len(names)==len(set(names))
    matrix_name=representation
    sparse.save_npz(OUT/"datasets"/f"{matrix_name}.npz",matrix)
    (OUT/"datasets"/f"{matrix_name}_columns.json").write_text(json.dumps(names,indent=2)+"\n")
    MATRIX_RECORDS[matrix_name]={"representations":parts,"block_weight":1.0,"shape":list(matrix.shape),
                               "file_sha256":hashlib.sha256((OUT/"datasets"/f"{matrix_name}.npz").read_bytes()).hexdigest()}
    return matrix,matrix_name


def fit_and_evaluate(representation,block_weight,C,matrix,matrix_name):
    name=f"{representation}_weight{block_weight:g}_C{C:g}".replace(".","p")
    template=SOURCE_MODELS[representation]
    model={**template,"C":C,"feature_block_weight":block_weight,"estimators":[],
           "training_matrix_file":f"datasets/{matrix_name}.npz","source_targets_file":"datasets/fixed_targets_weights.csv",
           "source_feature_model_files":[str((SOURCE/"models"/f"{part}.joblib").relative_to(DATA)) for part in (COMBINATIONS[representation] or ["embeddings_only"])]}
    # Preserve dense input for the embeddings-only control, as in the reference experiment.
    fit_matrix=matrix.toarray() if representation=="embeddings_only" else matrix
    with threadpool_limits(limits=1):
        for j in range(5):
            estimator=clone(template["estimators"][j]).set_params(C=C)
            included=W[:,j]>0
            normalized=W[included,j]*157/W[included,j].sum()
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                estimator.fit(fit_matrix[included],Y[included,j],sample_weight=normalized)
            model["estimators"].append(estimator)
    probabilities=predict_model(model,valid)
    if representation=="embeddings_only" and C==1000:
        np.testing.assert_allclose(probabilities,predict_model(template,valid),rtol=1e-8,atol=1e-10)
    predicted,empty=label_predictions(probabilities)
    sample=f1_score(truth,predicted,average="samples",zero_division=0)
    macro=f1_score(truth,predicted,average="macro",zero_division=0)
    if representation=="embeddings_only":BASELINE_SCORES[C]=sample
    known_predicted,_=label_predictions(predict_model(model,train.loc[known]))
    row={"experiment":name,"representation":representation,"feature_group":template["block"]["kind"],
         "additional_features":len(template["block"]["feature_names"]),"total_features":matrix.shape[1],
         "C":C,"block_weight":block_weight,"matrix_file":f"datasets/{matrix_name}.npz",
         "sample_f1":sample,"macro_f1":macro,"delta_vs_current_leader":sample-reference_sample,
         "delta_vs_same_C_embeddings":sample-BASELINE_SCORES[C],
         "known_training_sample_f1":f1_score(original_y[known],known_predicted,average="samples",zero_division=0),
         "fallback_trials":int(empty.sum()),"changed_trials_vs_current":int((predicted!=reference_prediction).any(axis=1).sum()),
         **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0))),
         "is_current":bool(representation=="embeddings_only" and C==1000),
         "distance_from_current_settings":abs(float(np.log10(C/1000)))+abs(float(np.log2(block_weight))),"kaggle_score":np.nan}
    ROWS.append(row)
    frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probabilities[:,j];frame[f"predicted_{r}"]=predicted[:,j]
    frame["used_fallback"]=empty
    frame["trial_f1"]=2*(truth*predicted).sum(axis=1)/(truth.sum(axis=1)+predicted.sum(axis=1))
    frame.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    joblib.dump(model,OUT/"models"/f"{name}.joblib",compress=3)
    pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
    return row


def run_representation(representation):
    weights=[1.0] if representation=="embeddings_only" else BLOCK_WEIGHTS
    for weight in weights:
        matrix,matrix_name=prepare_matrix(representation,weight)
        for C in C_GRID:fit_and_evaluate(representation,weight,C,matrix,matrix_name)
    subset=pd.DataFrame(ROWS).query("representation==@representation")
    best_row=subset.sort_values("sample_f1",ascending=False).iloc[0]
    print(f"{representation}: {len(subset)} configurations, best sample F1={best_row.sample_f1:.6f}, "
          f"macro F1={best_row.macro_f1:.6f}, C={best_row.C:g}, weight={best_row.block_weight:g}, "
          f"change vs current={best_row.delta_vs_current_leader:+.6f}",flush=True)

run_representation("embeddings_only")


embeddings_only: 2 configurations, best sample F1=0.910256, macro F1=0.811421, C=1000, weight=1, change vs current=+0.000000


In [3]:
for representation in REPRESENTATIONS:run_representation(representation)


diseases_criteria: 2 configurations, best sample F1=0.893162, macro F1=0.794790, C=1000, weight=1, change vs current=-0.017094


diseases_pathways: 2 configurations, best sample F1=0.871795, macro F1=0.772471, C=100, weight=1, change vs current=-0.038462


criteria_pathways: 2 configurations, best sample F1=0.865385, macro F1=0.745498, C=1000, weight=1, change vs current=-0.044872


diseases_criteria_pathways: 2 configurations, best sample F1=0.873932, macro F1=0.760207, C=1000, weight=1, change vs current=-0.036325


In [4]:
comparison=pd.DataFrame(ROWS);assert len(comparison)==10
ranked=comparison.sort_values(["sample_f1","is_current","additional_features","distance_from_current_settings","experiment"],
                             ascending=[False,False,True,True,True],kind="stable")
best=ranked.iloc[0]
best_by_representation=ranked.groupby("representation",sort=False).head(1)
best_by_group=ranked.groupby("feature_group",sort=False).head(1)
best_by_representation.to_csv(OUT/"best_by_representation.csv",index=False)
best_by_group.to_csv(OUT/"best_by_group.csv",index=False)
comparison.query("representation=='embeddings_only'").to_csv(OUT/"embedding_controls.csv",index=False)
print("Selected:",best.experiment,"sample F1:",best.sample_f1,"macro F1:",best.macro_f1)
display(best_by_representation[["representation","C","block_weight","additional_features","sample_f1","macro_f1","delta_vs_current_leader","delta_vs_same_C_embeddings"]].round(6))
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid)
selected_prediction,_=label_predictions(selected_probability)
# Reload and independently rescore every saved label set.
for row in comparison.itertuples(index=False):
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    predictions=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    probs=frame[[f"probability_{r}" for r in REASONS]].to_numpy()
    np.testing.assert_array_equal(label_predictions(probs)[0],predictions)
    np.testing.assert_allclose(f1_score(truth,predictions,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,predictions,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
    reloaded=joblib.load(OUT/"models"/f"{row.experiment}.joblib")
    np.testing.assert_allclose(predict_model(reloaded,valid),probs,rtol=1e-12,atol=1e-12)
    persisted_row=sparse.load_npz(OUT/row.matrix_file)
    expected_row=sparse.csr_matrix(transform_model(reloaded,train))
    diff=persisted_row-expected_row
    assert not diff.nnz or abs(diff.data).max()<1e-12
    if row.representation=="embeddings_only" and row.C==1000:
        np.testing.assert_allclose(probs,predict_model(SOURCE_MODELS[row.representation],valid),rtol=1e-8,atol=1e-10)
# Winner preprocessing must recreate its exact saved matrix.
persisted=sparse.load_npz(OUT/best.matrix_file)
expected=sparse.csr_matrix(transform_model(selected,train))
difference=persisted-expected
assert not difference.nnz or abs(difference.data).max()<1e-12
# Independently refit winner from persisted matrix and targets/weights.
fixed_replay=pd.read_csv(OUT/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed_replay.ID.tolist()==train.ID.tolist()
np.testing.assert_array_equal(fixed_replay[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(fixed_replay[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
fit_matrix=persisted.toarray() if selected["block"]["kind"]=="none" else persisted
replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=fixed_replay[r].to_numpy(dtype=int);w=fixed_replay[f"weight_{r}"].to_numpy();included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(fit_matrix[included],y[included],sample_weight=w[included]*157/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid))[0],selected_prediction)
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest
print("All 10 saved label sets rescored; original settings reproduced; winner preprocessing/refit and source checks passed.")


Selected: embeddings_only_weight1_C1000 sample F1: 0.9102564102564102 macro F1: 0.811421460284173


,representation,C,block_weight,additional_features,sample_f1,macro_f1,delta_vs_current_leader,delta_vs_same_C_embeddings
1,embeddings_only,1000.0,1.0,0,0.910256,0.811421,0.000000,0.000000
3,diseases_criteria,1000.0,1.0,2071,0.893162,0.794790,-0.017094,-0.017094
9,diseases_criteria_pathways,1000.0,1.0,2098,0.873932,0.760207,-0.036325,-0.036325
4,diseases_pathways,100.0,1.0,98,0.871795,0.772471,-0.038462,-0.017094
7,criteria_pathways,1000.0,1.0,2027,0.865385,0.745498,-0.044872,-0.044872


All 10 saved label sets rescored; original settings reproduced; winner preprocessing/refit and source checks passed.


In [5]:
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_model(selected,test))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"C_grid":C_GRID,"block_weights":[1.0],"combinations":COMBINATIONS,
          "configurations":10,"reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "source_experiment":"classifier_features","generated_labels_changed":False,"generated_weights_changed":False,
          "preprocessing_refitted":False,"features_added_to":"classifier only","classifier":"logistic regression",
          "embedding_scaler_fit_rows":157,"original_vocabulary_idf_fit_rows":1171,"validation_rows":78,"validation_used_for_fit":False,
          "fitting_weight_total_per_reason":157,"cutoff":0.5,"fallback":True,
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "candidate":candidate,"protected_input_sha256":protected_hashes,"matrices":MATRIX_RECORDS,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*')) if p.is_file()},
          "selection":"sample F1; ties prefer current leader, fewer features, then original C",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"scipy":scipy.__version__,"numpy":np.__version__,"pandas":pd.__version__},
          "new_kaggle_scores":None,"reported_best_kaggle":0.87393}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
def table(frame):
    lines=["| Representation | C | Added features | Sample F1 | Macro F1 | Change vs leader | Change vs same-C embeddings |",
           "|---|---:|---:|---:|---:|---:|---:|"]
    for row in frame.itertuples(index=False):
        lines.append(f"| {row.representation} | {row.C:g} | {row.additional_features} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_current_leader:+.6f} | {row.delta_vs_same_C_embeddings:+.6f} |")
    return "\n".join(lines)
lines=["# Combined classifier features with fixed DBSCAN pseudo-labels","",
       "Four combinations of diseases, criteria, and broad pathways are tested with logistic regression at C=100 and C=1000, "
       "with an embeddings-only control at each C: ten configurations. Each matrix includes the 150 embeddings exactly once.","",
       "Frozen representations: diseases_min5 (71 binary columns), criteria_min3 (2,000 TF-IDF columns), and pathways_min30 "
       "(26 broad pathway columns plus one missing-annotation flag). Disease and pathway choices are the smallest tied "
       "representations from the initial feature comparison; criteria_min3 is the strongest previous text representation. "
       "Every feature-block weight is 1. No vocabularies, IDF, scalers, targets, or pseudo-label weights are refitted. "
       "Cutoff 0.5, highest-probability fallback, and fitting-weight total 157 per reason remain fixed.","",
       f"Reference sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       f"Selected **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Results","",table(comparison),"",
       "## Verification and interpretation","",
       "Every saved model reloads with matching probabilities, and its preprocessing exactly reproduces its persisted "
       "combined matrix. All ten saved prediction sets independently reproduce their sample and macro F1. The original "
       "embeddings-only leader reproduces its previous probabilities, and the selected model independently refits from "
       "the saved matrix and shared targets/weights with matching probabilities. Protected source artifacts retain their hashes.","",
       "Matched-C controls separate feature-combination changes from changing regularization. Logistic regression adds "
       "feature contributions; these combinations introduce no explicit interaction terms. The same repeatedly consulted "
       "78 validation trials are used for selection, so the results do not establish independent test improvement.","",
       "datasets/ preserves each exact combined sparse matrix, ordered columns, and shared trial IDs/targets/weights. "
       "models/ persists all native sklearn preprocessing components and fitted classifiers, so these same combined inputs "
       "can be reused in a subsequent nonlinear-model experiment. comparison.csv, best_by_representation.csv, "
       "embedding_controls.csv, settings.json, and validation_predictions/ preserve results and checksums.","",
       "No nonlinear models, Kaggle submissions, commits, or tags were used in this round.",""]
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No configuration improves the primary validation metric; retain the current embeddings-only leader.")
summary="\n".join(lines)+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print(table(comparison))
print("Selected:",best.experiment,"Candidate:",candidate)


| Representation | C | Added features | Sample F1 | Macro F1 | Change vs leader | Change vs same-C embeddings |
|---|---:|---:|---:|---:|---:|---:|
| embeddings_only | 100 | 0 | 0.888889 | 0.790092 | -0.021368 | +0.000000 |
| embeddings_only | 1000 | 0 | 0.910256 | 0.811421 | +0.000000 | +0.000000 |
| diseases_criteria | 100 | 2071 | 0.888889 | 0.797110 | -0.021368 | -0.000000 |
| diseases_criteria | 1000 | 2071 | 0.893162 | 0.794790 | -0.017094 | -0.017094 |
| diseases_pathways | 100 | 98 | 0.871795 | 0.772471 | -0.038462 | -0.017094 |
| diseases_pathways | 1000 | 98 | 0.865385 | 0.745498 | -0.044872 | -0.044872 |
| criteria_pathways | 100 | 2027 | 0.863248 | 0.764743 | -0.047009 | -0.025641 |
| criteria_pathways | 1000 | 2027 | 0.865385 | 0.745498 | -0.044872 | -0.044872 |
| diseases_criteria_pathways | 100 | 2098 | 0.854701 | 0.746244 | -0.055556 | -0.034188 |
| diseases_criteria_pathways | 1000 | 2098 | 0.873932 | 0.760207 | -0.036325 | -0.036325 |
Selected: embeddings_only_weight1